In [1]:
import pandas as pd
import pyarrow.parquet as pq
import os

Define function to compare and merge similar columns

In [2]:
def compare_and_merge_columns(df, column_pairs, time_tolerance='1s'):
    """
    Compares given column pairs for equality or near equality.
    If the columns are datetime and within time_tolerance, merges them.
    Returns:
        cleaned_df: DataFrame with cleaned columns
        report: dictionary summarizing similarity stats
    """
    report = {}
    cleaned_df = df.copy()

    for col1, col2, base_name in column_pairs:
        if pd.api.types.is_datetime64_any_dtype(cleaned_df[col1]) and pd.api.types.is_datetime64_any_dtype(
                cleaned_df[col2]):
            # Handle datetime columns
            time_diff = (cleaned_df[col1] - cleaned_df[col2]).abs()
            close_match = time_diff <= pd.to_timedelta(time_tolerance)
            mismatch_rate = 1 - close_match.mean()
            cleaned_df[base_name] = cleaned_df[col1].where(close_match, cleaned_df[col2])
            cleaned_df.drop(columns=[col1, col2], inplace=True)
            report[base_name] = {
                'type': 'datetime',
                'close_match_ratio': round(close_match.mean(), 4),
                'mismatch_ratio': round(mismatch_rate, 4),
            }
        else:
            # Handle other columns
            exact_match = cleaned_df[col1] == cleaned_df[col2]
            mismatch_rate = 1 - exact_match.mean()
            if mismatch_rate < 0.01:  # >99% match, collapse to one
                cleaned_df[base_name] = cleaned_df[col1].where(exact_match, cleaned_df[col2])
                cleaned_df.drop(columns=[col1, col2], inplace=True)
            else:
                # Keep both if significantly different, but log mismatch rate
                report[base_name] = {
                    'type': 'other',
                    'exact_match_ratio': round(exact_match.mean(), 4),
                    'mismatch_ratio': round(mismatch_rate, 4),
                }

    return cleaned_df, report

In [3]:
# Load merged parquet file
merged_df = pd.read_parquet(
    r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_bookings_measurements.parquet")

In [4]:
# Drop exact duplicate rows
merged_df.drop_duplicates(inplace=True)

In [5]:
# Define column pairs to compare
column_pairs = [
    ('created_at_meas', 'created_at_book', 'created_at'),
    ('serial_number_id_meas', 'serial_number_id_book', 'serial_number_id'),
    ('station_id_meas', 'station_id_book', 'station_id')
]

In [6]:
# Compare and merge columns (with time tolerance for datetime)
merged_df, similarity_report = compare_and_merge_columns(merged_df, column_pairs, time_tolerance='500ms')

In [ ]:
# Optional: remove any remaining completely identical columns
merged_df = merged_df.loc[:, ~merged_df.T.duplicated()]

In [7]:
# Save cleaned version
cleaned_output_path = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_bookings_measurements_cleaned.parquet"
merged_df.to_parquet(cleaned_output_path, index=False)

print("✅ Cleaned merged file saved to:", cleaned_output_path)
print("📊 Similarity report:", similarity_report)
print("📏 Final shape:", merged_df.shape)

✅ Cleaned merged file saved to: M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_bookings_measurements_cleaned.parquet
📊 Similarity report: {'created_at': {'type': 'datetime', 'close_match_ratio': np.float64(0.9969), 'mismatch_ratio': np.float64(0.0031)}}
📏 Final shape: (43934134, 17)


In [8]:
merged_df.head()

,measure_step_number,measure_value,book_state_meas,part_number,lower_limit,upper_limit,measurement_name_encoded,measurement_unit_encoded,is_within_limits,book_state_book,workstep_number_mes,book_stamp,part_group,line_id,created_at,serial_number_id,station_id
0,273,4.11456,1,a13143e7,0.0,10.0,41832,2581331,1,1,2,2025-03-28 10:43:05.759000+00:00,83e223f1,ad63c958,2025-03-28 10:43:04.158000+00:00,32e85784,464416bb
1,863,828.67300,1,5a6867de,490.0,910.0,41833,13027095,1,1,2,2025-03-11 23:55:49.840000+00:00,8db45195,ad63c958,2025-03-11 23:55:49.788000+00:00,c89a5b10,464416bb
2,863,830.37200,1,5a6867de,490.0,910.0,41833,13027095,1,1,2,2025-03-17 21:03:22.216000+00:00,8db45195,ad63c958,2025-03-17 21:03:22.172000+00:00,4fdb5292,464416bb
3,863,830.42200,1,5a6867de,490.0,910.0,41833,13027095,1,1,2,2025-05-06 22:10:04.832000+00:00,8db45195,ad63c958,2025-05-06 22:10:04.760000+00:00,aa69f270,464416bb
4,863,829.61300,1,5a6867de,490.0,910.0,41833,13027095,1,1,2,2025-04-22 08:12:20.228000+00:00,8db45195,ad63c958,2025-04-22 08:12:20.181000+00:00,1d99775f,464416bb
